# Thesis PDF Figures — Publication-Ready Exports

Grouped delta charts + F1 by difficulty for each pairwise comparison.
Saved as PDF to `MT/data/thesis_figures/`. Self-contained (loads from Delta).

In [0]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib, os
from matplotlib.patches import Patch
matplotlib.rcParams['figure.dpi'] = 200
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.titlesize'] = 12
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.pad_inches'] = 0.15

df = spark.table("dev_forge_default.mt_davide.experiment_thesis_results").toPandas()
for c in ['answer_f1','answer_precision','answer_recall','numeric_hallucination_risk',
          'latency_seconds','total_tokens','claim_groundedness']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['abstained'] = df['abstained'].astype(str).isin(['True','true','1'])
df['answerable'] = df['answerable'].astype(bool)

out_dir = '/Workspace/Users/dquaglio@relias.com/MT/data/thesis_figures'
os.makedirs(out_dir, exist_ok=True)

diffs = ['easy','medium','hard','very_hard']
diff_labels = ['Easy','Medium','Hard','Very Hard']

def _get_val(arch, col):
    s = df[df['architecture'] == arch]
    if col == '__sql__':
        return (s['sql_execution_status'] == 'success').mean()
    return s[col].dropna().mean()

def plot_delta(ax, a1, a2, metrics):
    """Grouped horizontal delta chart. metrics = [(group_label, name, col, higher_better), ...]."""
    names, deltas, higher_better, group_starts = [], [], [], []
    prev_group = None
    for group, name, col, hb in metrics:
        v1, v2 = _get_val(a1, col), _get_val(a2, col)
        pct = ((v2 - v1) / abs(v1) * 100) if v1 != 0 else 0
        names.append(name); deltas.append(pct); higher_better.append(hb)
        if group != prev_group:
            group_starts.append((len(names) - 1, group))
            prev_group = group
    n = len(names)
    y_pos = []; gap = 0
    for i in range(n):
        for gs_idx, gs_label in group_starts:
            if gs_idx == i and i > 0: gap += 0.5
        y_pos.append(i + gap)
    y_pos = np.array(y_pos)
    colors = []
    for d, hb in zip(deltas, higher_better):
        if hb: colors.append('#4CAF50' if d > 0 else '#EF5350')
        else:  colors.append('#4CAF50' if d < 0 else '#EF5350')
    bars = ax.barh(y_pos, deltas, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5, height=0.55)
    for bar, d in zip(bars, deltas):
        offset = 3 if d >= 0 else -3; ha = 'left' if d >= 0 else 'right'
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2, f'{d:+.1f}%', ha=ha, va='center', fontsize=9, fontweight='bold')
    for gs_idx, gs_label in group_starts:
        yy = y_pos[gs_idx]
        if gs_idx > 0:
            mid = (y_pos[gs_idx - 1] + yy) / 2
            ax.axhline(mid, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
        ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else -5, yy + 0.7, gs_label, fontsize=8, fontstyle='italic', color='#555555', ha='left', va='bottom')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_yticks(y_pos); ax.set_yticklabels(names, fontsize=10); ax.set_xlabel('Change (%)', fontsize=10)
    max_abs = max(abs(min(deltas)), abs(max(deltas)), 30)
    ax.set_xlim(-max_abs * 1.4, max_abs * 1.4); ax.set_ylim(y_pos[0] - 0.5, y_pos[-1] + 1.2); ax.grid(axis='x', alpha=0.3)
    ax.legend(handles=[Patch(facecolor='#4CAF50', label='Improvement'), Patch(facecolor='#EF5350', label='Degradation')], fontsize=8, loc='lower right')
    ax.invert_yaxis()

print('\u2713 Shared setup loaded: df, plot_delta, out_dir')

In [0]:
# =============================================
# COMPARISON 1: SAS → SAS+RAG
# =============================================
fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1, 0.8]})
fig1.suptitle('Comparison 1: SAS vs SAS+RAG \u2014 Effect of Retrieval Augmentation', fontsize=13, fontweight='bold', y=1.02)

metrics_c1 = [
    ('Answer Quality', 'Precision',     'answer_precision',              True),
    ('Answer Quality', 'Recall',        'answer_recall',                 True),
    ('Safety',         'Groundedness',  'claim_groundedness',            True),
    ('Safety',         'Hallucination', 'numeric_hallucination_risk',    False),
    ('SQL',            'SQL Rate',      '__sql__',                       True),
    ('Efficiency',     'Latency',       'latency_seconds',               False),
    ('Efficiency',     'Tokens',        'total_tokens',                  False),
]
plot_delta(ax1a, 'SAS', 'SAS_RAG', metrics_c1)
ax1a.set_title('Percentage Change (SAS \u2192 SAS+RAG)', fontsize=11, fontweight='bold')

w = 0.35; x = np.arange(len(diffs))
for i, (arch, lbl, col) in enumerate([('SAS','SAS','#2196F3'), ('SAS_RAG','SAS+RAG','#00BCD4')]):
    vals = [df[(df['architecture']==arch)&(df['difficulty']==d)]['answer_f1'].mean() for d in diffs]
    bars = ax1b.bar(x + i*w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals): ax1b.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1b.set_xticks(x + w/2); ax1b.set_xticklabels(diff_labels, fontsize=10); ax1b.set_ylabel('F1 Score', fontsize=10)
ax1b.set_title('F1 by Difficulty Level', fontsize=11, fontweight='bold'); ax1b.legend(fontsize=9); ax1b.set_ylim(0, 0.32); ax1b.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig1.savefig(f'{out_dir}/comp1_sas_vs_sas_rag.pdf', format='pdf')
plt.show()
print(f'\u2713 Saved: {out_dir}/comp1_sas_vs_sas_rag.pdf')

In [0]:
# =============================================
# COMPARISON 2: SAS+RAG → MAS+RAG
# =============================================
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1, 0.8]})
fig2.suptitle('Comparison 2: SAS+RAG vs MAS+RAG \u2014 Effect of Multi-Agent Coordination', fontsize=13, fontweight='bold', y=1.02)

metrics_c2 = [
    ('Answer Quality', 'F1',            'answer_f1',                     True),
    ('Answer Quality', 'Precision',     'answer_precision',              True),
    ('Answer Quality', 'Recall',        'answer_recall',                 True),
    ('Safety',         'Groundedness',  'claim_groundedness',            True),
    ('Safety',         'Hallucination', 'numeric_hallucination_risk',    False),
    ('SQL',            'SQL Rate',      '__sql__',                       True),
    ('Efficiency',     'Latency',       'latency_seconds',               False),
    ('Efficiency',     'Tokens',        'total_tokens',                  False),
]
plot_delta(ax2a, 'SAS_RAG', 'MAS_RAG', metrics_c2)
ax2a.set_title('Percentage Change (SAS+RAG \u2192 MAS+RAG)', fontsize=11, fontweight='bold')

bar_labels = ['SAS+RAG', 'MAS+RAG']; bar_x = np.arange(2); bar_w = 0.5
bar_data = []
for a in ['SAS_RAG', 'MAS_RAG']:
    sub = df[df['architecture'] == a]
    unans = sub[sub['answerable'] == False]; ans = sub[sub['answerable'] == True]
    correct_abstain = int(unans['abstained'].sum()); missed_unans = len(unans) - correct_abstain
    false_abstain = int(ans['abstained'].sum()); answered = len(ans) - false_abstain
    pff = sub['post_failure_fabrication'].astype(str).isin(['True','true','1']).sum(); fab_pct = pff / len(sub) * 100
    bar_data.append({'correct': correct_abstain, 'missed': missed_unans, 'false_abs': false_abstain, 'answered': answered, 'fab_pct': fab_pct})

categories = ['Correct Abstention','Missed (should abstain)','False Abstention','Answered']
cat_keys = ['correct','missed','false_abs','answered']; cat_colors = ['#66BB6A','#EF5350','#FFA726','#42A5F5']
bottom = np.zeros(2)
for cat, key, col in zip(categories, cat_keys, cat_colors):
    vals = np.array([d[key] for d in bar_data])
    ax2b.bar(bar_x, vals, bar_w, bottom=bottom, label=cat, color=col, edgecolor='white', linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 2: ax2b.text(i, b + v / 2, str(int(v)), ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    bottom += vals
for i, d in enumerate(bar_data): ax2b.text(i, bottom[i] + 1.5, f'Fabrication: {d["fab_pct"]:.0f}%', ha='center', fontsize=8, fontstyle='italic', color='#C62828')
ax2b.set_xticks(bar_x); ax2b.set_xticklabels(bar_labels, fontsize=11); ax2b.set_ylabel('Number of Runs (out of 75)', fontsize=10)
ax2b.set_title('Abstention Behavior Breakdown', fontsize=11, fontweight='bold')
ax2b.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=2, framealpha=0.9)
ax2b.set_ylim(0, 92); ax2b.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig2.savefig(f'{out_dir}/comp2_sas_rag_vs_mas_rag.pdf', format='pdf')
plt.show()
print(f'\u2713 Saved: {out_dir}/comp2_sas_rag_vs_mas_rag.pdf')

In [0]:
# =============================================
# COMPARISON 3: MAS+RAG → Dynamic
# =============================================
fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1, 0.8]})
fig3.suptitle('Comparison 3: MAS+RAG vs Dynamic \u2014 Effect of Dynamic Filtering', fontsize=13, fontweight='bold', y=1.02)

metrics_c3 = [
    ('Answer Quality', 'Precision',     'answer_precision',              True),
    ('Answer Quality', 'Recall',        'answer_recall',                 True),
    ('Safety',         'Groundedness',  'claim_groundedness',            True),
    ('Safety',         'Hallucination', 'numeric_hallucination_risk',    False),
    ('SQL',            'SQL Rate',      '__sql__',                       True),
    ('Efficiency',     'Latency',       'latency_seconds',               False),
    ('Efficiency',     'Tokens',        'total_tokens',                  False),
]
plot_delta(ax3a, 'MAS_RAG', 'DYNAMIC_FILTERED_MAS_RAG', metrics_c3)
ax3a.set_title('Percentage Change (MAS+RAG \u2192 Dynamic)', fontsize=11, fontweight='bold')

w = 0.35; x = np.arange(len(diffs))
for i, (arch, lbl, col) in enumerate([('MAS_RAG','MAS+RAG','#FF9800'), ('DYNAMIC_FILTERED_MAS_RAG','Dynamic','#9C27B0')]):
    vals = [df[(df['architecture']==arch)&(df['difficulty']==d)]['answer_f1'].mean() for d in diffs]
    bars = ax3b.bar(x + i*w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals): ax3b.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax3b.set_xticks(x + w/2); ax3b.set_xticklabels(diff_labels, fontsize=10); ax3b.set_ylabel('F1 Score', fontsize=10)
ax3b.set_title('F1 by Difficulty Level', fontsize=11, fontweight='bold')

mas_fa = df[(df['architecture']=='MAS_RAG')&(df['answerable']==True)]['abstained'].mean()*100
dyn_fa = df[(df['architecture']=='DYNAMIC_FILTERED_MAS_RAG')&(df['answerable']==True)]['abstained'].mean()*100
mas_ca = df[(df['architecture']=='MAS_RAG')&(df['answerable']==False)]['abstained'].mean()*100
dyn_ca = df[(df['architecture']=='DYNAMIC_FILTERED_MAS_RAG')&(df['answerable']==False)]['abstained'].mean()*100
textstr = (f'False Abstention:\n  MAS+RAG: {mas_fa:.0f}%\n  Dynamic: {dyn_fa:.0f}%\n\n'
           f'Correct Abstention (unans.):\n  MAS+RAG: {mas_ca:.0f}%\n  Dynamic: {dyn_ca:.0f}%')
ax3b.text(0.98, 0.98, textstr, transform=ax3b.transAxes, fontsize=8, verticalalignment='top', horizontalalignment='right',
          bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray', alpha=0.9))
ax3b.legend(fontsize=9); ax3b.set_ylim(0, 0.52); ax3b.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig3.savefig(f'{out_dir}/comp3_mas_rag_vs_dynamic.pdf', format='pdf')
plt.show()
print(f'\u2713 Saved: {out_dir}/comp3_mas_rag_vs_dynamic.pdf')